In [8]:
import duckdb
import pandas as pd
from pathlib import Path
import diskcache
from itertools import chain
from collections import Counter

import igraph as ig
import matplotlib.pyplot as plt
import seaborn.objects as so
import seaborn as sns

from utils.pandas_setup import pandas_setup
pandas_setup()

import pyalex
from pyalex import Works, Authors, Sources, Institutions, Topics, Publishers, Funders
pyalex.config.email = "Lawrence.Cram@anu.edu.au"
pyalex.config.max_retries = 0
pyalex.config.retry_backoff_factor = 0.1
pyalex.config.retry_http_codes = [429, 500, 503]

MY_DATA_PATH = Path('../DATA/')
MY_DATABASE_FILE = Path(MY_DATA_PATH / 'econ.duckdb')
MY_CACHE_FILE = Path('/home/lc/m/.cache/econommicsbusiness/cache.db')
DATAFILES_PATH = Path('../DATAFILES')

In [9]:
class SetUp:

    def __init__(self):
        self._setup_db()
        self._setup_cache()
        return
    
    def _setup_db(self):
        self.db = duckdb.connect(MY_DATABASE_FILE)
        self.db.sql("ATTACH IF NOT EXISTS ':memory:'")
        self.db.sql(""" SET memory_limit = '56GB';
                        SET threads = 6;
                        SET preserve_insertion_order = false;
                        SET order_by_non_integer_literal=true;
                        SET enable_progress_bar = true;
                        SET temp_directory = '/home/lc/m/.tmp';
                    """)
        for tab in ['econ.edge_list_sources', 'econ.edge_list_institutions', 'econ.edge_list_both']: 
            self.db.sql(f"DROP TABLE IF EXISTS {tab}")
        self.db.sql("SHOW ALL TABLES").show()
        return
    
    def _setup_cache(self):
        self.cache = diskcache.Cache(MY_CACHE_FILE, size_limit=16_000_000_000)
        print(f'{self.cache.check() = }')
        print(f'{self.cache.volume() = }')
        return
    
    def name_of_global_obj(self, obj=None):
        for objname, oid in globals().items():
            if oid is obj:
                return objname
            

### This class constructs the pageRank for sources linked by citation counts

-  build the edge list of journal (citer) -> journal (cited)  
-  construct an iGraph from the edge list  
-  run pageRank  
-  for each author, calculate the number of ciations and the number of citations weighted by the pageRank of the citing journal



In [10]:
class PageRanks(SetUp):

    def __init__(self):
        super().__init__()
        return

    def vertex_labels(self):
        sql = """
            SELECT DISTINCT source_id AS id,
                    source_name AS name
                FROM works
            UNION
            SELECT DISTINCT institution_id AS id,
                    institution_name AS name
                from authorships
            """
        df = self.db.sql(sql).df()
        self.vertex_labels = dict(zip(df.id, df.name))
        return
    
    def make_edge_lists(self):
        # SQL code to assemble the edge list with the count of work-work citations as weights
        sql = """
            CREATE OR REPLACE TABLE econ.edge_list_combined AS

                WITH 
                source_cte AS
                    (SELECT w.work_id, w.source_id, 1.0 AS weight FROM
                    (SELECT count(work_id) AS counts, source_id FROM econ.works GROUP BY ALL) sub
                    LEFT JOIN econ.works w
                    USING (source_id)
                    WHERE sub.counts > 100
                    ORDER BY sub.counts DESC
                    ),
                institution_part_cte AS
                    (SELECT institution_id, 1.0 AS weight FROM
                    (SELECT count(work_id) AS counts, institution_id FROM
                        (SELECT DISTINCT work_id, institution_id FROM econ.authorships)
                        GROUP BY ALL
                    )
                    WHERE counts > 100 AND institution_id NOT NULL
                    ORDER BY counts DESC
                    ),
                institution_weight_cte AS
                        (SELECT DISTINCT a.work_id, i.institution_id, 1.0/count(i.institution_id) AS weight
                        FROM institution_part_cte i
                        LEFT JOIN econ.authorships a
                        ON a.institution_id = i.institution_id
                        WHERE a.institution_id NOT NULL
                        GROUP BY ALL
                        ),
                institution_cte AS
                        (SELECT a.work_id, iw.institution_id, iw.weight
                        FROM institution_weight_cte iw
                        LEFT JOIN econ.authorships a
                        USING (work_id)
                        ),
                works_sources_institutions AS
                    (
                    SELECT DISTINCT work_id, source_id AS item, weight
                    FROM source_cte
                    UNION
                    SELECT DISTINCT work_id, institution_id AS item, weight
                    FROM institution_cte
                    ORDER BY work_id ASC, item ASC
                    ),
                citations AS
                    (SELECT work_id AS citer_id, unnest(referenced_works) AS cited_id FROM econ.cited),
                full_table AS
                    (SELECT citer_id, w1.item AS citer_item, w1.weight AS citer_weight,
                        cited_id, w2.item AS cited_item, w2.weight AS cited_weight 
                    FROM citations c
                        LEFT JOIN works_sources_institutions w1
                        ON c.citer_id = w1.work_id
                        LEFT JOIN works_sources_institutions w2
                        ON c.cited_id = w2.work_id
                        WHERE citer_item NOT NULL AND cited_item NOT NULL
                    ORDER BY citer_id, citer_item, citer_weight, cited_id)

                -- SELECT * FROM citations
                -- SELECT * FROM source_cte
                -- SELECT count(DISTINCT work_id), count(DISTINCT source_id) FROM source_cte
                -- SELECT * FROM institution_cte
                -- SELECT count(DISTINCT institution_id) FROM institution_part_cte
                -- SELECT count(DISTINCT work_id), count(DISTINCT institution_id) FROM institution_cte
                -- SELECT * FROM works_sources_institutions

                SELECT citer_item, cited_item, sum(citer_weight*cited_weight) AS weights -- Institutions weighted at both citer and cited ends
                FROM full_table
                GROUP BY citer_item, cited_item
        """
        self.db.sql(sql)
        return

    def _extract_edge_list(self):
        match self.kind:
            case 'sources':
                cut = '/S'
            case 'institutions':
                cut = '/I'
            case 'both':
                cut = 'http'
        return self.db.sql(f"SELECT * FROM econ.edge_list_combined WHERE contains(citer_item, '{cut}') = true AND contains(cited_item, '{cut}') = true").df()

    def construct_graph(self, kind=None):
        self.kind = kind
        df_edges = self._extract_edge_list()
# DROP SELF LOOPS
        mask = [citer != cited for citer, cited in zip(df_edges.citer_item, df_edges.cited_item)]
        df_edges = df_edges[mask]
        print("THIS MODEL EXCLUDES SELF_LOOPS FROM THE GRAPH")
    
        vertices = set(df_edges.citer_item.tolist() + df_edges.cited_item.tolist())
        vs = pd.DataFrame(data=list(vertices), columns=['id'])
        vs['label'] = [self.vertex_labels.get(id) for id in vs.id]
        print(f'{vs['id'].nunique() = } {df_edges.shape = }\n{df_edges.head()}\n{df_edges.tail()}')
        print(f'GRAPH NODES - Number of sources and institutions {vs.shape = }\n{vs.head()}')
        self.g = ig.Graph.DataFrame(df_edges, directed=True, use_vids=False, vertices=vs)
        summary = ig.summary(self.g, verbosity=1, width=256, edge_list_format='auto', max_rows=2, print_graph_attributes=True, 
                                          print_vertex_attributes=True, print_edge_attributes=True, full=False)
        print(f'*** SUMMARY OF self.g\n{summary}')
        return
    
    def run_pagerank(self):
        print('pageRank')
        damping = 0.5 # if self.kind == 'both' else 0.85
        print(f'>> RUN pagerank with {damping = } for {self.kind = }')
        ranks = self.g.pagerank(damping=damping, weights='weights', implementation='prpack')
        print(f'>> CHECK pageRank  - should sum to unity {sum(ranks) = } then scaled to a sum of 100 as in "eigenfactor" score')
        sum_ranks = sum(ranks)
        ranks = [100.0*r/sum_ranks for r in ranks]
        self.pagerank = pd.DataFrame(zip(ranks, self.g.vs["name"], self.g.vs['label'], 
                                         self.g.strength(mode='in', weights=None), #'weights'),
                                         self.g.strength(mode='out', weights=None)), #'weights')),
                                      columns=['pageRank','citer', 'label', 'in_degree', 'out_degree']).sort_values('pageRank', ascending=False)
        self._append_influence()
        df = self.pagerank
        print(f'{df.shape = }\n{df.head()}')
        self.db.sql(f"CREATE OR REPLACE TABLE econ.pagerank_{self.kind} AS SELECT * FROM df")
        return
    
    def _append_influence(self):
        print(f'_append influence {self.kind = }')
        if self.kind == "sources":
            self._append_influence_sources()
        elif self.kind == 'institutions':
            self._append_influence_institutions()
        return
    
    def _append_influence_sources(self):
        sql = """
            WITH total_works_cte AS
                (SELECT count(DISTINCT work_id) AS total_works FROM econ.works)
            SELECT DISTINCT p.pageRank, w.source_id, pageRank*(SELECT * FROM total_works_cte)/count(work_id)/100 AS influence
                    FROM econ.pagerank_sources p
                    LEFT JOIN econ.works w
                    ON p.citer = w.source_id
                    GROUP BY p.pageRank, w.source_id
        """
        df = self.db.sql(sql).df()
        print(f'{df.shape = }\n{df.head()}')
        dd = dict(zip(df.source_id, df.influence))
        self.pagerank['influence'] = [dd.get(institution) for institution in self.pagerank.citer]        
        return    
    
    def _append_influence_institutions(self):
        sql = """
            WITH 
                total_works_cte AS
                    (SELECT count(DISTINCT work_id) AS total_works FROM econ.works),
                work_institution_cte AS
                    (SELECT DISTINCT work_id, institution_id 
                    FROM econ.works w
                    LEFT JOIN econ.authorships a
                    USING (work_id))
            SELECT DISTINCT institution_id, pageRank*(SELECT * FROM total_works_cte)/count(work_id)/100 AS influence
                FROM econ.pagerank_institutions p
                LEFT JOIN work_institution_cte w
                ON p.citer = w.institution_id
                GROUP BY pageRank, institution_id
        """
        df = self.db.sql(sql).df()
        print(f'{df.shape = }\n{df.head()}')
        dd = dict(zip(df.institution_id, df.influence))
        self.pagerank['influence'] = [dd.get(institution) for institution in self.pagerank.citer]
        return
    
    def report_pagerank(self):
        kind = self.kind
        print(f'report pagerank {kind = }')
        df = self.db.sql(f"SELECT * FROM econ.pagerank_{kind}").df().sort_values('pageRank', ascending=False).reset_index(drop=True)        
        print(f'{df.shape = }\n{df.head()}')
        print(f'Sum of out_degrees {df['out_degree'].sum() = }')
        print(f'Sum of pageranks {df['pageRank'].sum() = }')
        self.db.sql("SELECT count(DISTINCT work_id) AS original_works_count FROM works").show()
        return
    
    def run_reputation_both(self):
        df = self.db.sql("SELECT * FROM econ.pagerank_both").df()
        df_s = self.db.sql("SELECT * FROM econ.pagerank_sources").df()
        print(f'{df_s.shape = }\n{df_s.head()}')
        df_i = self.db.sql("SELECT * FROM econ.pagerank_institutions").df()
        print(f'{df_i.shape = }\n{df_i.head()}')
        dd = dict(zip(df_s.citer, df_s.influence)) | dict(zip(df_i.citer, df_i.influence))
        df['influence'] = [dd.get(item) for item in df.citer]
        print(f'{df.shape = }\n{df.head()}')
        self.db.sql("CREATE OR REPLACE TABLE econ.pagerank_both AS (SELECT * FROM df)")
        return

In [11]:

class WeightedCitationCounts(SetUp):

    def __init__(self):
         super().__init__()
         return
    
    def make_weighted_citations(self):

        for kind in ['sources', 'institutions', 'both']:
            sql = f"""
                CREATE OR REPLACE TABLE econ.weighted_citations_{kind} AS
                    WITH 
                    item_cte AS
                        (SELECT DISTINCT citer_item
                        FROM econ.edge_list_combined
                        ), -- cte to filter combined list
                    source_cte AS
                        (SELECT citer_item AS source_id
                        FROM item_cte
                        WHERE contains(citer_item, '/S') = true
                        ), -- cte to filter sources from combined list
                    institution_cte AS
                        (SELECT citer_item AS institution_id
                        FROM item_cte
                        WHERE contains(citer_item, '/I') = true
                        ), -- cte to filter institutions     p = Plotters()
                    works_sources_institutions AS
                        (
                        SELECT DISTINCT w.work_id, w.source_id, a.institution_id
                        FROM econ.works w
                        LEFT JOIN econ.authorships a
                        USING (work_id)
                        LEFT JOIN source_cte s
                        ON s.source_id = w.source_id
                        LEFT JOIN institution_cte i
                        ON i.institution_id = a.institution_id
                        WHERE a.institution_id NOT NULL AND w.source_id NOT NULL
                        ORDER BY w.source_id, a.institution_id
                        ), -- cte to build a table of the sources (one-to-one) and institutions (one-to-many) for each work 
                    work_reputation_sources AS
                        (
                        SELECT DISTINCT w.work_id, influence AS reputation -- pageRank AS reputation
                        FROM works_sources_institutions w
                        LEFT JOIN pagerank_sources p
                        ON w.source_id = p.citer
                        ), -- cte to attach source-only pageranks to works
                    work_reputation_institutions AS
                        (
                        SELECT DISTINCT w.work_id, influence AS reputation -- sum(pageRank)/count(pagerank) AS reputation
                        FROM works_sources_institutions w
                        LEFT JOIN pagerank_institutions p
                        ON w.institution_id = p.citer
                        GROUP BY ALL
                        ), -- cte to attach institution-only averaged pagerank to works
                    work_reputation_both AS
                        (SELECT sub.work_id, 0.5*(sub.reputation + p1.influence) AS reputation -- pageRank
                        FROM
                            (
                                SELECT w.work_id, w.source_id, influence AS reputation -- sum(pageRank)/count(pageRank) AS reputation
                                FROM works_sources_institutions w
                                LEFT JOIN pagerank_both p
                                ON w.institution_id = p.citer
                                GROUP BY ALL
                            ) sub
                            LEFT JOIN pagerank_both p1
                            ON sub.source_id = p1.citer
                        ), -- cte to attach averaged (source and average institution) pagerank to works
                    citer_cited AS
                        (
                        SELECT work_id AS citer_id, unnest(referenced_works) AS cited_id
                        FROM cited
                        ), -- cte to make citer-cited relation
                    citers_authors AS
                        (
                        SELECT DISTINCT author_id, c.citer_id
                        FROM econ.authorships a
                        LEFT JOIN citer_cited c
                        ON a.work_id = c.cited_id
                        )  -- cte to build a list of citer works for each author 
                    
                    --SELECT * FROM item_cte
                    -- SELECT * FROM works_sources_institutions
                    -- SELECT * FROM citer_cited
                    -- SELECT * FROM citers_authors
                    -- SELECT * FROM work_pagerank_sources
                    -- SELECT * FROM work_pagerank_institutions
                    -- SELECT * FROM work_pagerank_both

                    SELECT author_id, count(DISTINCT citer_id) AS citer_count, sum(reputation) AS citer_count_weighted
                    FROM
                        (SELECT c.citer_id, c.author_id, reputation
                        FROM citers_authors c
                        LEFT JOIN work_reputation_{kind} w
                        ON c.citer_id = w.work_id
                        WHERE reputation NOT NULL)
                    GROUP BY ALL
                    ORDER BY citer_count DESC

            """
            self.db.sql(sql)
            self.db.sql(f"SELECT * FROM econ.weighted_citations_{kind} ORDER BY citer_count DESC").show()
        return



In [12]:
class Plotters(SetUp):

    def __init__(self):
        super().__init__()
        return

    def plot_pagerank(self):
        hold = []
        for kind in ['sources', 'institutions', 'both']:
            temp = self.db.sql(f"SELECT * FROM pagerank_{kind}").df().sort_values('pageRank', ascending=True).reset_index(drop=True).reset_index(drop=False)
            temp['panel'] = kind
            hold.append(temp)
        df = pd.concat(hold, axis=0)
        df = df.melt(id_vars=['out_degree', 'panel'], value_vars=['pageRank', 'influence'], var_name='measure', value_name='Value')
        print(f'{df.shape = }\n{df.head()}')
        g = sns.relplot(df, x='out_degree', y='Value', hue='measure', col='panel', kind='scatter')
        for ax in g.axes.flat:
            ax.set_xlim((10, 10000))
            ax.set_ylim((0.01, 10))
            ax.set_xscale('log')
            ax.set_yscale('log')
        plt.suptitle("PageRank and Influence versus out-degree (i.e. emitted citations)", fontsize=16)
        plt.tight_layout(rect=[0, 0.03, 1, 0.95])
        plt.show()
        return

    def plot_model(self):

        df = self.summary[['author_id', 'citations_endogenous', 
                           'reputation_sources', 'reputation_institutions', 'reputation_both', 
                           'hca_endogenous', 'hca_total', '2yr_mean_citedness', 'h_index', 'group']]
        print(f'{df.shape = }\n{df.head()}')
        df['pointsize'] = [0.01 if s == 'X' else 5.0 if s == 'C' else 15 for s in df['group']]
        df = df.sort_values(['group', 'pointsize'], ascending=[False, True])
        hold = []
        for kind in ['sources', 'institutions', 'both']:
            temp = df.copy()
            temp['reputation'] = temp[f'reputation_{kind}']
            temp['panel'] = kind
            hold.append(temp)
        df_in = pd.concat(hold, axis=0)
        self._plot_reputations(df_in=df_in)
        self._plot_hca(df_in=df_in)

        # df = df[df.group != 'X']
        # df = df[['citer_count_weighted', 'citer_count', 'ratio', 'author_name', 'group']].sort_values('ratio', ascending=False).reset_index(drop=True)
        # df.to_csv(f'../DATA/weighted_citations_{kind}.csv')

        return

    def _plot_reputations(self, df_in=None):

        print(df_in.info())
        df = df_in.melt(id_vars=['panel', 'author_id', 'citations_endogenous', 'group', 'pointsize'], 
                        value_vars=['reputation'], 
                        value_name='Value', 
                        var_name='measure')
        print(f'{df.shape = }\n{df.head()}')      
        g = sns.relplot(df, x='citations_endogenous', y='Value', hue='group', size='pointsize', col='panel', kind='scatter', alpha=0.5)
        for ax in g.axes.flat:
            ax.set_xscale('log')
            ax.set_yscale('log')
        g.set(xlabel='Endogenous citation count per author', ylabel="Author's reputation score")
        plt.suptitle("Reputation versus citations", fontsize=16)
        sns.move_legend(g, 'upper right')
        plt.tight_layout(rect=[0, 0.03, 1, 0.95])
        plt.show()
        return

    def _plot_hca(self, df_in=None):

        for kind in ['hca_total', 'hca_endogenous', 'h_index', '2yr_mean_citedness']:

            df = df_in.melt(id_vars=['panel', 'author_id', kind, 'group', 'pointsize'], 
                            value_vars=['reputation'], 
                            value_name='Value', 
                            var_name='measure')
            print(f'{df.shape = }\n{df.head()}')    
            g = sns.relplot(df, x=kind, y='Value', hue='group', size='pointsize', col='panel', kind='scatter', alpha=0.5)
            for ax in g.axes.flat:
                ax.set_xscale('log')
                ax.set_yscale('log')
            g.set(xlabel=f"Author's {kind.replace('_', ' ').upper()}", ylabel="Author's reputation score")
            plt.suptitle(f"{kind.replace('_', ' ').upper()} versus Reputation", fontsize=16)
            sns.move_legend(g, 'upper right')
            plt.tight_layout(rect=[0, 0.03, 1, 0.95])
            plt.show()
        return
        
    def extract_data(self):

        self.summary = self.db.sql("SELECT * FROM econ.citation_summary").df()\
            [['author_id', 'author_name', 'works_count_endogenous', 'citations_endogenous', 'hca_total', 
             'hca_endogenous', 'works_count_total', 'cited_by_count', '2yr_mean_citedness', 'h_index']]

        for kind in ['sources', 'institutions', 'both']:
            temp = self.db.sql(f"SELECT * FROM econ.weighted_citations_{kind}").df().sort_values('citer_count_weighted', ascending=False)
            temp['reputation'] = temp.citer_count_weighted/temp.citer_count
            dd = dict(zip(temp.author_id, temp.reputation))
            self.summary[f'reputation_{kind}'] = [dd.get(a) for a in self.summary.author_id]

        sample = self.db.sql("SELECT * FROM econ.sample_names").df()
        dd_group = dict(zip(sample.author_id, sample.Group))
        self.summary['group'] = [dd_group.get(aid, 'X') for aid in self.summary.author_id]
        
        summary = self.summary.rename(columns={"2yr_mean_citedness": "citedness"})
        print(f'{self.summary.shape = }\n{self.summary.head()}')
        self.db.sql("SELECT * FROM summary").show()
        self.db.sql("CREATE OR REPLACE TABLE econ.summary AS (SELECT * FROM summary)")
        return


In [13]:
def main():

    # pr = PageRanks()
    # pr.vertex_labels()
    # pr.make_edge_lists()
    # for kind in ['sources', 'institutions', 'both']:
    #     pr.construct_graph(kind=kind)
    #     pr.run_pagerank()
    #     pr.report_pagerank()
    # pr.run_reputation_both()

    # cn = WeightedCitationCounts()
    # cn.make_weighted_citations()

    p = Plotters()
    p.extract_data()
    # p.plot_pagerank()
    # p.plot_model()

In [14]:
if __name__ == "__main__":
    main()
    print("DONE!")

┌──────────┬─────────┬───────────────────────────────────────┬──────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────┬───────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────